# 6BUIS017W CRM & CM with Business Intelligence – Individual Coursework (CWK1)
## Task 1: Data Preparation and Feature Engineering
**Market Analysis and Stock Segmentation Using RFM**

IIT ID: 20231076 | UoW ID: 20521398

**Static dataset:** the S&P 500 list changes over time, so the data was downloaded once (list retrieved on the date saved in `data/sp500_constituents.csv`) and saved in the repository. With `USE_SNAPSHOT = True` this notebook loads that saved data, so every run uses the same shares and prices. Set it to `False` to scrape Wikipedia and download from Yahoo Finance again.

## Setup


In [3]:
# Install the packages (versions are recorded for reproducibility)
!pip install -q yfinance==0.2.66 lxml

In [4]:
import os, sys, json, time
import datetime as dt
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Study settings (from the coursework brief) ----
START = '2021-01-01'      # first date requested
END = '2025-01-01'        # end date is EXCLUSIVE (last possible trading day = 31 Dec 2024)
BENCH = '^GSPC'           # S&P 500 benchmark index
WINDOW = 60               # latest 60 trading sessions for Frequency and Monetary
BASELINE = 20             # preceding 20 sessions for the volume baseline
MULT = 2                  # high-volume day: volume > 2 x baseline mean

USE_SNAPSHOT = True       # True = load the saved static dataset, False = download again

GITHUB_USER = 'shahlyfayeek'
REPO = 'crm-cwk1-stock-segmentation'
SNAPSHOT_URL = 'https://raw.githubusercontent.com/' + GITHUB_USER + '/' + REPO + '/main/data/'

DATA_DIR = 'data'
FIG_DIR = 'figures'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Package versions (required by the brief)
versions = {
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'yfinance': yf.__version__,
    'seaborn': sns.__version__,
    'matplotlib': __import__('matplotlib').__version__,
}
print(versions)

def snapshot_path(name):
    """Use the file in the Colab data folder if it is there, otherwise read it from the GitHub repo."""
    local = DATA_DIR + '/' + name
    if os.path.exists(local):
        return local
    return SNAPSHOT_URL + name

{'python': '3.13.18', 'pandas': '2.2.3', 'numpy': '2.4.6', 'yfinance': '0.2.66', 'seaborn': '0.13.2', 'matplotlib': '3.10.0'}


## 1.1 Ticker selection and data acquisition

**Choices**
* Ticker list: current S&P 500 constituents table from Wikipedia, read with `pd.read_html` (as in Week 4).
* Price source: **`yfinance`** (chosen over YahooFinancials because it was used in the Week 1–5 notebooks and downloads many tickers at once).
* Prices are downloaded with `auto_adjust=False` so I keep both `Close` (split-adjusted) and `Adj Close` (split- and dividend-adjusted) plus `Volume`.
* Study period: 1 Jan 2021 to 1 Jan 2025, end date exclusive.

In [5]:
# --- 1.1a  S&P 500 ticker list ---
if USE_SNAPSHOT:
    sp500 = pd.read_csv(snapshot_path('sp500_constituents.csv'))
    retrieval_date = sp500['Retrieval_Date'].iloc[0]
    print('Loaded the saved ticker list, retrieved on:', retrieval_date)
else:
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    # A User-Agent is needed, otherwise Wikipedia can return HTTP 403 (Week 4 notebook)
    table = pd.read_html(url, storage_options={'User-Agent': 'Mozilla/5.0'})
    sp500 = table[0].copy()

    retrieval_date = dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%d')
    print('Constituent list retrieved on:', retrieval_date, '(UTC)')

    # Yahoo uses '-' instead of '.' (BRK.B -> BRK-B)
    sp500['Ticker'] = sp500['Symbol'].str.replace('.', '-', regex=False)
    sp500['Retrieval_Date'] = retrieval_date
    sp500 = sp500[['Ticker', 'Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Retrieval_Date']]

print('Number of tickers on the list:', len(sp500))
print('Duplicate tickers on the list:', sp500['Ticker'].duplicated().sum())
sp500.to_csv(DATA_DIR + '/sp500_constituents.csv', index=False)
sp500.head()

Loaded the saved ticker list, retrieved on: 2026-11-14
Number of tickers on the list: 503
Duplicate tickers on the list: 0


,Ticker,Symbol,Security,GICS Sector,GICS Sub-Industry,Retrieval_Date
0,MMM,MMM,3M,Industrials,Industrial Conglomerates,2026-11-14
1,AOS,AOS,A. O. Smith,Industrials,Building Products,2026-11-14
2,ABT,ABT,Abbott Laboratories,Health Care,Health Care Equipment,2026-11-14
3,ABBV,ABBV,AbbVie,Health Care,Biotechnology,2026-11-14
4,ACN,ACN,Accenture,Information Technology,IT Consulting & Other Services,2026-11-14


In [6]:
# --- 1.1b  Daily prices and volume (shares + benchmark) ---
tickers = sp500['Ticker'].tolist()
all_symbols = tickers + [BENCH]

if USE_SNAPSHOT:
    adj_raw = pd.read_csv(snapshot_path('raw_adj_close.csv.gz'), index_col=0, parse_dates=True)
    close_raw = pd.read_csv(snapshot_path('raw_close.csv.gz'), index_col=0, parse_dates=True)
    vol_raw = pd.read_csv(snapshot_path('raw_volume.csv.gz'), index_col=0, parse_dates=True)
    source_used = 'saved snapshot of the Yahoo Finance download (list retrieved ' + retrieval_date + ')'
else:
    # Download in chunks of 100 tickers to reduce Yahoo rate-limit problems
    parts = []
    for i in range(0, len(all_symbols), 100):
        chunk = all_symbols[i:i + 100]
        parts.append(yf.download(chunk, start=START, end=END, auto_adjust=False, progress=False))
        time.sleep(1)
    raw = pd.concat(parts, axis=1)
    adj_raw = raw['Adj Close']
    close_raw = raw['Close']
    vol_raw = raw['Volume']
    source_used = 'live Yahoo Finance download'

for table_ in (adj_raw, close_raw, vol_raw):
    table_.index = pd.to_datetime(table_.index)
    table_.sort_index(inplace=True)

print('Data source used:', source_used)
print('Adj Close shape (dates x symbols):', adj_raw.shape)
print('First date:', adj_raw.index.min().date(), '| Last date:', adj_raw.index.max().date())

Data source used: saved snapshot of the Yahoo Finance download (list retrieved 2026-11-14)
Adj Close shape (dates x symbols): (1005, 504)
First date: 2021-01-04 | Last date: 2024-12-31


In [7]:
# --- 1.1c  Save the snapshots (static dataset) ---
adj_raw.to_csv(DATA_DIR + '/raw_adj_close.csv.gz', compression='gzip')
close_raw.to_csv(DATA_DIR + '/raw_close.csv.gz', compression='gzip')
vol_raw.to_csv(DATA_DIR + '/raw_volume.csv.gz', compression='gzip')

# Which symbols have no data at all?
failed = [t for t in all_symbols if (t not in adj_raw.columns) or adj_raw[t].isna().all()]
print('Symbols with no data at all:', failed)
pd.Series(failed, name='Symbol', dtype=str).to_csv(DATA_DIR + '/download_failures.csv', index=False)

Symbols with no data at all: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']


## 1.2 Data validation and alignment

**Validation rules (a share is retained only if it passes all of them):**

| # | Rule | Why |
|---|---|---|
| R1 | Data downloaded | no data = nothing to analyse |
| R2 | Same history as the benchmark (first price on the first benchmark date) | unequal histories (new listings) give different numbers of weeks |
| R3 | No missing Adj Close, Close or Volume on any benchmark trading date | missing prices are **not filled** (the Week 4/5 `ffill/bfill` approach creates artificial zero returns and uses future prices) |
| R4 | All prices > 0 | non-positive prices are errors |
| R5 | Volume > 0 on every date | negative or zero volume for an S&P 500 share is a data error |

Duplicate dates and corporate actions (splits, dividends) are checked separately below.

In [8]:
# --- 1.2a  Benchmark and date checks ---
bench_adj = adj_raw[BENCH].dropna()
bench_dates = bench_adj.index
print('Benchmark trading days:', len(bench_dates))
print('Duplicate dates in the price table:', adj_raw.index.duplicated().sum())
print('Duplicate dates in the volume table:', vol_raw.index.duplicated().sum())
print('Dates outside the study period:', ((adj_raw.index < START) | (adj_raw.index >= END)).sum())
print('Dates not in the benchmark calendar:', len(adj_raw.index.difference(bench_dates)))
print('Benchmark non-positive prices:', (bench_adj <= 0).sum())

# Use the benchmark calendar for every share (identical dates)
adj = adj_raw.reindex(bench_dates)
close = close_raw.reindex(bench_dates)
vol = vol_raw.reindex(bench_dates)

Benchmark trading days: 1005
Duplicate dates in the price table: 0
Duplicate dates in the volume table: 0
Dates outside the study period: 0
Dates not in the benchmark calendar: 0
Benchmark non-positive prices: 0
